# Cross Dataset Evaluation

Before running the notebook, update the configuration values to match your environment.    
Choose one external preprocessed benchmark for cross-dataset binary evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import sys
import glob
import os
import random

import cv2
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from torchvision import transforms
from tqdm import tqdm


#############################################################################
# Configuration
# Update the configuration below before running this notebook.
# Change TEST_DATASET_NAME to evaluate another preprocessed benchmark.
#############################################################################

PROJECT_ROOT = Path('/content/drive/MyDrive/DE-Fake-it')
DATA_ROOT = PROJECT_ROOT / 'Dataset'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'

MODEL_NAME = 'EfficientNet-b0'
CHECKPOINT_NAME = 'checkpoint_v35'
BEST_CHECKPOINT_NAME = CHECKPOINT_NAME + '_best'
RANDOM_SEED = 42

BASE_PATH = str(DATA_ROOT)
CHECKPOINT_PATH = str(CHECKPOINT_DIR / CHECKPOINT_NAME)

TEST_DATASET_NAME = 'celeb-df'  # 'celeb-df', 'deeperForensics'

TEST_INPUT_FILE_PATH = str(DATA_ROOT / TEST_DATASET_NAME / '*')
TEST_METADATA_PATH = str(DATA_ROOT / TEST_DATASET_NAME)
meta_data_path = TEST_METADATA_PATH

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Configuration")
print(f"Model: {MODEL_NAME}")
print(f"Checkpoint name: {BEST_CHECKPOINT_NAME}")
print(f"Dataset: {TEST_DATASET_NAME}")

### Prepare Metadata
Create the metadata file required for external dataset evaluation.

In [ ]:
import importlib.util

MAKE_METADATA_SCRIPT = PROJECT_ROOT / 'model' / 'Helpers' / 'make_meta_data.py'
metadata_spec = importlib.util.spec_from_file_location('make_meta_data', MAKE_METADATA_SCRIPT)
make_meta_data = importlib.util.module_from_spec(metadata_spec)
metadata_spec.loader.exec_module(make_meta_data)

METADATA_INPUT_GLOB = str(DATA_ROOT / TEST_DATASET_NAME / '**' / '*.mp4')
metadata_df, metadata_csv_path, metadata_excel_path = make_meta_data.build_metadata(
    input_glob=METADATA_INPUT_GLOB,
    base_dir=str(DATA_ROOT),
    output_dir=TEST_METADATA_PATH,
)

print('Metadata Configuration')
print(f'Input glob: {METADATA_INPUT_GLOB}')
print(f'Output CSV: {metadata_csv_path}')
print(f'Output Excel: {metadata_excel_path}')


## Load Model For Evaluation

The trained checkpoint and frame transform are loaded before running benchmark inference.


In [ ]:
from model.research.metrics import compute_eer, compute_pauc
from model.research.modeling import ResearchModel as Model
from model.research.device import get_device
from model.research.visualization import save_binary_confusion_matrix, save_roc_curve

device = get_device()
print(f"Device: {device}")

# Model is imported from model.research.modeling.
model = Model(num_binary_classes=2, num_method_classes=7, model_name=MODEL_NAME).to(device)
model.load_state_dict(torch.load(f'{CHECKPOINT_PATH}/{BEST_CHECKPOINT_NAME}.pt', map_location=device))
model.eval()

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


## Run Video-Level Prediction

Each test video is evaluated frame by frame and aggregated into a video-level binary prediction.


In [ ]:
new_video_files = glob.glob(f'{TEST_INPUT_FILE_PATH}/*.mp4')
random.shuffle(new_video_files)

results = []
label_list = []
video_bin_scores = []

with torch.no_grad():
    for video_path in tqdm(new_video_files):
        cap = cv2.VideoCapture(video_path)
        frame_preds = []
        frame_scores = []
        frame_idx = 0

        relative_path = os.path.relpath(video_path, BASE_PATH).replace("\\", "/")

        if 'real' in relative_path.lower():
            label = 'REAL'
        elif 'fake' in relative_path.lower():
            label = 'FAKE'
        else:
            label = 'unknown'
        label_list.append(label)

        success, frame = cap.read()
        while success:
            frame_idx += 1
            if frame_idx % 1 == 0:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                input_tensor = transform(frame)
                input_tensor = input_tensor.unsqueeze(0).unsqueeze(0)
                input_tensor = input_tensor.to(device).float()

                _, output_bin, _ = model(input_tensor)
                _, predicted_bin = torch.max(output_bin, 1)

                score = torch.softmax(output_bin.squeeze(0), dim=0)[1].item()
                frame_scores.append(score)
                frame_preds.append(predicted_bin.item())

            success, frame = cap.read()

        if frame_scores:
            video_bin_scores.append(np.mean(frame_scores))

        cap.release()
        final_prediction = 'Unknown' if len(frame_preds) == 0 else ('REAL' if round(sum(frame_preds)/len(frame_preds)) == 1 else 'FAKE')
        result = {
            'Filename': os.path.basename(video_path),
            'Filepath': video_path,
            'label': label,
            'Prediction': final_prediction,
        }
        results.append(result)


## Save Benchmark Reports

Predictions, confusion matrix, ROC curve, AUC, EER, and pAUC are saved or printed for the selected external dataset.


In [ ]:
output_excel_path = f"{CHECKPOINT_PATH}/(test)_{BEST_CHECKPOINT_NAME}_predictions_{TEST_DATASET_NAME}.xlsx"
df = pd.DataFrame(results)
df.to_excel(output_excel_path, index=False, engine='openpyxl')

print(f"✅ 모든 비디오 예측 결과가 엑셀로 저장되었습니다: {output_excel_path}")

y_true = label_list
y_pred = [r['Prediction'] for r in results]
true_bin = [0 if l == 'FAKE' else 1 for l in y_true]
pred_bin = [0 if p == 'FAKE' else 1 for p in y_pred]

print("\n================ External Benchmark Report ================")
save_binary_confusion_matrix(true_bin, pred_bin, CHECKPOINT_PATH, BEST_CHECKPOINT_NAME, suffix=f"plot(test_{TEST_DATASET_NAME})")

save_roc_curve(
    torch.tensor(true_bin),
    torch.tensor(video_bin_scores),
    CHECKPOINT_PATH,
    f"{BEST_CHECKPOINT_NAME}",
    suffix=f"roc_curve(test_{TEST_DATASET_NAME})",
    score_mode="probability",
    print_auc=False,
    print_suffix=None,
)

# FAKE를 positive class (1)로 보기 위해 점수 뒤집기
video_fake_scores = [1 - s for s in video_bin_scores]
true_bin = [1 if l == 'FAKE' else 0 for l in y_true]

auc_val = roc_auc_score(true_bin, video_fake_scores)
eer, eer_threshold = compute_eer(true_bin, video_fake_scores)
pauc = compute_pauc(true_bin, video_fake_scores)

print(f"AUC (FAKE=1 기준): {auc_val:.4f}")
print(f"EER (FAKE=1 기준): {eer:.4f} at threshold {eer_threshold:.4f}")
print(f"pAUC@0.1 (FAKE=1 기준): {pauc:.4f}")
